# 第二章：文档加载与索引构建

## 学习目标

- 使用 `SimpleDirectoryReader` 从文件加载文档
- 理解 Document → Node 的转换过程
- 掌握 `SentenceSplitter` 文本切分策略
- 用 `VectorStoreIndex` 一行代码构建 RAG
- 对比 LangChain 的 RAG 构建流程

## 0. 环境准备

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")

from llama_index.llms.openai_like import OpenAILike
from llama_index.embeddings.openai_like import OpenAILikeEmbedding
from llama_index.core import Settings

# 初始化 LLM（通过 OpenAI 兼容接口接入 Qwen）
llm = OpenAILike(
    model="qwen-plus",
    api_base=os.environ["Qwen_API_BASE"],
    api_key=os.environ["Qwen_API_KEY"],
    is_chat_model=True,
)
Settings.llm = llm

# 切换为 GLM（同样是 OpenAI 兼容接口）：
# from llama_index.llms.openai_like import OpenAILike
# Settings.llm = OpenAILike(model="glm-4-plus", api_base=os.environ["GLM_API_BASE"], api_key=os.environ["GLM_API_KEY"], is_chat_model=True)

# 初始化 Embedding 模型
Settings.embed_model = OpenAILikeEmbedding(
    model_name="text-embedding-v3",
    api_base=os.environ["Qwen_API_BASE"],
    api_key=os.environ["Qwen_API_KEY"],
)

print("环境初始化完成")

## 1. 手动创建 Document 对象

In [ ]:
from llama_index.core import Document

# 准备示例文档（关于 Python 的知识卡片）
documents = [
    Document(
        text="Python 是一种解释型、面向对象的高级编程语言。它由 Guido van Rossum 于 1991 年发创。Python 的设计哲学强调代码的可读性和简洁性。",
        metadata={"topic": "Python 简介"},
    ),
    Document(
        text="列表（list）是 Python 中最常用的数据结构之一。它是有序的、可变的集合，支持索引访问、切片、追加和删除操作。列表用方括号 [] 表示。",
        metadata={"topic": "列表"},
    ),
    Document(
        text="字典（dict）是 Python 中的键值对数据结构。它是无序的（Python 3.7+ 保持插入顺序），键必须是不可变类型。字典用花括号 {} 表示。",
        metadata={"topic": "字典"},
    ),
    Document(
        text="装饰器（decorator）是 Python 中的一种设计模式，用于在不修改函数代码的情况下扩展函数功能。使用 @decorator 语法糖，本质上是高阶函数。",
        metadata={"topic": "装饰器"},
    ),
    Document(
        text="生成器（generator）是 Python 中用于惰性求值的工具。通过 yield 关键字定义，每次调用 next() 返回一个值，节省内存。适合处理大数据集。",
        metadata={"topic": "生成器"},
    ),
]

print(f"共创建 {len(documents)} 个文档")
for doc in documents:
    print(f"  [{doc.metadata['topic']}] {doc.text[:30]}...")

### 刚才发生了什么？

`Document` 是 LlamaIndex 中最基本的数据单元。每个 `Document` 包含：

- **`text`**：文档的正文内容
- **`metadata`**：附加的元数据字典（如来源、主题、日期等）

和 LangChain 的 `Document(page_content=..., metadata=...)` 非常相似，只是字段名不同：LangChain 用 `page_content`，LlamaIndex 用 `text`。

## 2. SimpleDirectoryReader：从目录加载文档

In [ ]:
import os

# 创建示例文件
os.makedirs("./sample_docs", exist_ok=True)

with open("./sample_docs/python_intro.txt", "w") as f:
    f.write("Python 是一种解释型编程语言，广泛应用于 Web 开发、数据分析、人工智能等领域。\n")
    f.write("Python 的语法简洁优雅，被称为'可执行的伪代码'。")

with open("./sample_docs/langchain_intro.txt", "w") as f:
    f.write("LangChain 是一个用于开发 LLM 应用的框架。\n")
    f.write("它提供了链式编排、Agent、RAG 等功能，帮助开发者快速构建 AI 应用。")

print("示例文件已创建")

In [ ]:
from llama_index.core import SimpleDirectoryReader

# 从目录加载所有文档
reader = SimpleDirectoryReader("./sample_docs")
file_documents = reader.load_data()

print(f"加载了 {len(file_documents)} 个文档")
for doc in file_documents:
    print(f"  来源: {doc.metadata.get('file_name', 'unknown')}")
    print(f"  内容: {doc.text[:50]}...")
    print()

### 刚才发生了什么？

`SimpleDirectoryReader` 是 LlamaIndex 最常用的文档加载器。它会：

1. 扫描指定目录下的所有文件
2. 自动识别文件类型（`.txt`、`.pdf`、`.md`、`.docx` 等）
3. 读取内容并封装为 `Document` 对象
4. **自动填充元数据**（`file_name`、`file_path`、`file_type` 等）

对比 LangChain 的文档加载方式：

| LangChain | LlamaIndex |
|---|---|
| `TextLoader("file.txt")` 加载单个文件 | `SimpleDirectoryReader("dir/")` 加载整个目录 |
| 需要不同 Loader 对应不同格式 | 自动识别文件类型 |
| 返回 `Document(page_content=...)` | 返回 `Document(text=...)` |
| 元数据需手动设置 | 自动填充 file_name, file_path 等元数据 |

一个目录、一行代码，就能加载所有支持的文件格式——这是 LlamaIndex 追求的「开箱即用」风格。

## 3. 从 Document 到 Node：文本切分

在 LlamaIndex 中，文档不会直接进入索引。它们会先被切分为更小的单元——**Node（节点）**。

为什么要切分？因为：
- LLM 的上下文窗口有限，不能把整本书塞进去
- 向量检索需要细粒度的文本块，才能精准匹配
- 更小的块 = 更精确的检索 + 更少的噪音

In [ ]:
from llama_index.core.node_parser import SentenceSplitter

# 使用句子分割器
splitter = SentenceSplitter(chunk_size=100, chunk_overlap=20)

# 将文档切分为节点
nodes = splitter.get_nodes_from_documents(documents)

print(f"5 个文档 → {len(nodes)} 个节点")
print()
for i, node in enumerate(nodes):
    print(f"节点 {i}:")
    print(f"  文本: {node.text[:60]}...")
    print(f"  元数据: {node.metadata}")
    print(f"  节点ID: {node.node_id[:8]}...")
    print()

### 刚才发生了什么？

`SentenceSplitter` 按句子边界将文档切分为节点。与简单按字符数切分不同，它会尽量在完整句子处断开，避免把一句话切成两半。

关键参数：

| 参数 | 说明 | 默认值 |
|------|------|--------|
| `chunk_size` | 每个节点的最大 token 数 | 1024 |
| `chunk_overlap` | 相邻节点的重叠 token 数 | 200 |

对比 LangChain 的文本切分：

| LangChain | LlamaIndex |
|---|---|
| `RecursiveCharacterTextSplitter` | `SentenceSplitter` |
| 按字符数切分，递归尝试分隔符 | 按句子边界切分 |
| 返回 `Document` 列表 | 返回 `TextNode` 列表 |
| 独立于索引的步骤 | 可集成到索引构建流程中 |

注意：由于我们的示例文档本身就很短（每个不到 100 token），所以切分后的节点数可能和文档数一样。在真实场景中，一个长文档会被切成多个节点。

## 4. 节点之间的关系

In [ ]:
# 查看节点之间的关系
node = nodes[0]
print(f"节点类型: {type(node).__name__}")
print(f"节点关系: {node.relationships}")

### 刚才发生了什么？

每个 `TextNode` 都有一个 `relationships` 字典，记录了它与其他节点以及源文档的关系。常见的关系类型：

- **SOURCE**：指向原始的 `Document`（这个节点是从哪个文档切出来的）
- **PREVIOUS**：前一个节点（同一文档内的上文）
- **NEXT**：后一个节点（同一文档内的下文）

这是 LlamaIndex 的一个重要设计：**节点知道自己的上下文**。在检索时，可以利用这些关系获取更多上下文信息，而不仅仅是孤立的文本块。

这一点 LangChain 默认不提供——切分后的 `Document` 之间没有内置的前后关系。

## 5. VectorStoreIndex：一行代码构建 RAG

In [ ]:
# ❌ LangChain 的 RAG 构建（4 个步骤）
# from langchain.text_splitter import RecursiveCharacterTextSplitter
# from langchain_openai import OpenAIEmbeddings
# from langchain_community.vectorstores import FAISS
#
# splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
# chunks = splitter.split_documents(documents)
# embeddings = OpenAIEmbeddings(...)
# vectorstore = FAISS.from_documents(chunks, embeddings)
# retriever = vectorstore.as_retriever()
print("LangChain 需要 4 步：加载 → 切分 → 嵌入 → 存储")

In [ ]:
# ✅ LlamaIndex 一行代码搞定
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex.from_documents(documents)
print(f"索引构建完成！类型: {type(index).__name__}")

### 刚才发生了什么？

`VectorStoreIndex.from_documents()` 这一行代码，内部完成了三件事：

1. **切分**：将 Document 切分为 Node（使用 `Settings` 中的默认分割器）
2. **嵌入**：调用 `Settings.embed_model` 将每个 Node 转为向量
3. **存储**：将向量存入内存中的向量存储

这就是 LlamaIndex 的设计哲学：**常见操作应该尽可能简单**。当你只需要一个标准的 RAG 流程时，一行代码就够了；当你需要更多控制时，可以拆开每一步手动配置（后面会演示）。

## 6. 第一次 RAG 查询

In [ ]:
# 创建查询引擎并提问
query_engine = index.as_query_engine()
response = query_engine.query("Python 的列表有什么特点？")
print(response)

### 刚才发生了什么？

`as_query_engine()` 创建了一个 `QueryEngine`。调用 `query()` 时，内部经历了以下步骤：

1. **Embed**：将问题「Python 的列表有什么特点？」转为向量
2. **Retrieve**：在索引中找到最相似的节点（默认返回 top-2）
3. **Synthesize**：将检索到的节点和原始问题一起发给 LLM，生成回答

整个流程就是一个完整的 RAG（Retrieval-Augmented Generation）管线。

## 7. 查看检索结果的细节

In [ ]:
# 查看检索到了哪些节点
response = query_engine.query("什么是装饰器？")
print("回答:", response)
print()
print("--- 检索到的源节点 ---")
for i, node in enumerate(response.source_nodes):
    print(f"节点 {i} (相似度: {node.score:.4f}):")
    print(f"  {node.text[:80]}...")
    print()

### 刚才发生了什么？

`response.source_nodes` 包含了检索阶段返回的所有节点，每个节点附带一个 `score`（相似度分数）。

这是调试 RAG 的关键工具：

- **相似度分数高但回答不好** → 可能是 LLM 的问题，需要调整 prompt
- **相似度分数低** → 检索没找到相关内容，需要改进切分策略或增加文档
- **检索到了错误的节点** → chunk_size 可能不合适，或者文档内容有歧义

养成习惯：每次调试 RAG 时，先看 `source_nodes`，再看最终回答。

## 8. 手动构建流水线（理解内部机制）

`VectorStoreIndex.from_documents()` 虽然方便，但有时我们需要更精细的控制。下面手动拆解每一步，让你看清内部发生了什么。

In [ ]:
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import VectorStoreIndex
from llama_index.core.ingestion import IngestionPipeline

# 显式构建流程（理解 from_documents 内部做了什么）
pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(chunk_size=200, chunk_overlap=20),
        Settings.embed_model,
    ]
)

nodes = pipeline.run(documents=documents)
print(f"Pipeline 产出 {len(nodes)} 个节点")

In [ ]:
# 用节点构建索引
index2 = VectorStoreIndex(nodes)
response = index2.as_query_engine().query("Python 的设计哲学是什么？")
print(response)

### 刚才发生了什么？

`IngestionPipeline` 让你显式定义数据处理的每一步：

1. **`SentenceSplitter`**：将文档切分为节点
2. **`Settings.embed_model`**：为每个节点生成嵌入向量

然后用 `VectorStoreIndex(nodes)` 直接从已处理的节点构建索引（注意这里不是 `from_documents`，而是直接传入节点列表）。

什么时候需要手动构建？

- 想要自定义切分策略（不同类型文档用不同参数）
- 需要在切分后做额外处理（如过滤、去重）
- 需要缓存中间结果（`IngestionPipeline` 支持缓存，避免重复 embedding）

### 常见问题

- **`VectorStoreIndex.from_documents()` 很慢**：每个节点都要调用 Embedding API。文档多时考虑批量处理或本地 Embedding 模型。
- **检索结果不相关**：`chunk_size` 太大或太小都会影响质量。建议先用默认值，再根据效果调整。
- **`Settings.embed_model` 未设置**：会尝试使用 OpenAI 默认端点，导致连接错误。务必在第一个代码单元格中配置好。
- **节点太多导致费用高**：每个节点都会调用一次 Embedding API。可以先用少量文档测试，确认效果后再扩大规模。

## 9. 清理示例文件

In [ ]:
# 清理示例文件
import shutil
if os.path.exists("./sample_docs"):
    shutil.rmtree("./sample_docs")
    print("已清理示例文件")

## 总结

本章学习了：

- ✅ 使用 `Document` 对象表示文档数据
- ✅ `SimpleDirectoryReader` 从目录加载文档
- ✅ `SentenceSplitter` 将文档切分为节点（Node）
- ✅ Document → Node → Index 的数据流转
- ✅ `VectorStoreIndex.from_documents()` 一行代码构建 RAG
- ✅ `as_query_engine()` 创建查询引擎并提问
- ✅ 对比 LangChain 的 4 步 RAG vs LlamaIndex 的 1 步

## 下一章预告

**第三章：查询引擎与响应合成** —— `query()` 内部到底发生了什么？学习如何控制检索参数、选择不同的响应合成策略（compact / refine / tree_summarize），以及自定义提示词模板。